# scPoli校正流程 Python版

把`原始基因表达h5ad`和`cell×10 scPoli h5ad`合并，按Seurat代码逻辑做epithelial subset、Normalize、UMAP过滤、scPoli聚类、local purity过滤、marker DotPlot和统计输出。

In [1]:
import os,numpy as np,pandas as pd,scanpy as sc,scipy.sparse as sp,matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
sc.settings.verbosity=2
sc.settings.seed=1234
np.random.seed(1234)

In [2]:
raw_h5ad="../../7_mapping_new_data/0511_rename_noIAISR/output_human/Atlas_allhuman_with_measure_allgeneID-inner-counts-annot.h5ad"##所有人类数据 原始counts + annot,现在这个文件中的cell_type_level1是ground truth
scpoli_h5ad="../../7_mapping_new_data/0507_no_IAISR/output_human/level1_human_atlas_nogene_noIAISR_2.h5ad"### all human scpoli data without gene names, but with annotation
outdir="./output_allhuman"
counts_layer="rounded_corrected_counts"
original_celltype_col="cell_type_level1"##原始文件里的标签
os.makedirs(outdir,exist_ok=True)

In [ ]:
#####读取文件 按 adata_scpoli 的细胞ID 对齐 adata_raw
adata_raw=sc.read_h5ad(raw_h5ad)
adata_scpoli=sc.read_h5ad(scpoli_h5ad)
print(adata_raw)
print(adata_raw.obs_names)
print(adata_scpoli)
print(adata_scpoli.obs_names)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 1019138 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species'
    var: 'original_gene_names', 'ensembl_id'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Index(['AAACCCAAGAGGTTAT-1', 'AAACCCAAGCAACAAT-1', 'AAACCCAAGGAGTCTG-1',
       'AAACCCACAACTCATG-1', 'AAACCCACAGCTCTGG-1', 'AAACCCAGTCAAGGCA-1',
       'AAACCCAGTCAATCTG-1', 'AAACCCAGTCACCCTT-1', 'AAACCCAGTGCATTAC-1',
       'AAACCCAGTGCGGCTT-1',
       ...
       'ATAACGCAGTACGCCC_H_plaque', 'CACATAGGTGTCTGAT_H_plaque',
       'CCATTCGAGTAGCCGA_H_plaque', 'GAATAAGAGAAAGTGG_H_plaque',
       'GACCAATCATGGTAGG_H_plaque', 'GACCTGGGTCT

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
###得到共同的细胞
adata_raw.obs_names=adata_raw.obs_names.astype(str)
adata_scpoli.obs_names=adata_scpoli.obs_names.astype(str)
##保留原始barcode
adata_raw.obs["barcode_for_match"]=adata_raw.obs_names
adata_scpoli.obs["barcode_for_match"]=adata_scpoli.obs_names.str[:-2]
##用 sample + barcode 构建唯一ID
adata_raw.obs["match_id"]=adata_raw.obs["sample"].astype(str)+"__"+adata_raw.obs["barcode_for_match"].astype(str)
adata_scpoli.obs["match_id"]=adata_scpoli.obs["sample"].astype(str)+"__"+adata_scpoli.obs["barcode_for_match"].astype(str)
##如果sample仍不唯一，就改成 dataset + sample + barcode
if adata_raw.obs["match_id"].duplicated().sum()>0 or adata_scpoli.obs["match_id"].duplicated().sum()>0:
    adata_raw.obs["match_id"]=adata_raw.obs["dataset"].astype(str)+"__"+adata_raw.obs["sample"].astype(str)+"__"+adata_raw.obs["barcode_for_match"].astype(str)
    adata_scpoli.obs["match_id"]=adata_scpoli.obs["dataset"].astype(str)+"__"+adata_scpoli.obs["sample"].astype(str)+"__"+adata_scpoli.obs["barcode_for_match"].astype(str)
print("raw duplicated match_id:",adata_raw.obs["match_id"].duplicated().sum())
print("scpoli duplicated match_id:",adata_scpoli.obs["match_id"].duplicated().sum())
if adata_raw.obs["match_id"].duplicated().sum()>0:
    raise ValueError("adata_raw match_id仍有重复，不能安全匹配")
if adata_scpoli.obs["match_id"].duplicated().sum()>0:
    raise ValueError("adata_scpoli match_id仍有重复，不能安全匹配")
adata_raw.obs_names=adata_raw.obs["match_id"].astype(str)
adata_scpoli.obs_names=adata_scpoli.obs["match_id"].astype(str)

##空值填充为query
if "atlas_key" not in adata_scpoli.obs:
    adata_scpoli.obs["atlas_key"]="query"
else:
    adata_scpoli.obs["atlas_key"]=adata_scpoli.obs["atlas_key"].astype("object")
    adata_scpoli.obs.loc[adata_scpoli.obs["atlas_key"].isna(),"atlas_key"]="query"
    adata_scpoli.obs.loc[adata_scpoli.obs["atlas_key"].astype(str).isin(["","nan","None","NA"]),"atlas_key"]="query"

print(adata_scpoli.obs["atlas_key"].value_counts(dropna=False))
##以 scpoli 为准，从 raw 中取相同细胞
scp_ids=adata_scpoli.obs_names
missing=scp_ids[~scp_ids.isin(adata_raw.obs_names)]
print("scpoli cells:",len(scp_ids))
print("missing in raw:",len(missing))
if len(missing)>0:
    print(missing[:50].tolist())
    raise ValueError("有 scpoli 细胞在 raw 中找不到")
adata=adata_raw[scp_ids,:].copy()
adata_com_scpoli=adata_scpoli.copy()
assert np.all(adata.obs_names==adata_com_scpoli.obs_names)
print("aligned adata:",adata)
print("aligned adata_com_scpoli:",adata_com_scpoli)

raw duplicated match_id: 0
scpoli duplicated match_id: 0
atlas_key
query    548208
ref      470566
Name: count, dtype: int64
scpoli cells: 1018774
missing in raw: 0
aligned adata: AnnData object with n_obs × n_vars = 1018774 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id'
    var: 'original_gene_names', 'ensembl_id'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
aligned adata_com_scpoli: AnnData object with n_obs × n_vars = 1018774 × 10
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_conta

In [ ]:
adata_query = adata[adata.obs['atlas_key'] == 'query']
adata_query.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   183446
B cell                    56046
Macrophage                55696
Natural killer cell       51294
Fibroblast                49419
Smooth muscle cell        42831
Endothelial cell          30615
Dendritic cell            24663
Monocyte                  16448
Neutrophil                14157
Pericyte                   9864
Basophil                   6162
Mast cell                  4427
Erythrocyte/Erythroid      3140
Name: count, dtype: int64

In [ ]:
adata_query = adata[adata.obs['atlas_key'] == 'ref']
adata_query.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   154319
Neutrophil                68634
Macrophage                48793
Fibroblast                39718
Endothelial cell          34734
B cell                    34289
Smooth muscle cell        31685
Natural killer cell       20001
Monocyte                  16440
Dendritic cell             9879
Pericyte                   9121
Mast cell                  2532
Erythrocyte/Erythroid       302
Basophil                    119
Name: count, dtype: int64

In [ ]:
adata.obs['atlas_key'].value_counts()

atlas_key
query    548208
ref      470566
Name: count, dtype: int64

In [ ]:
adata_com_scpoli.write("./output_allhuman/allhuman_scpoli_com.h5ad")
adata.write("./output_allhuman/allhuman_raw_com.h5ad")

In [ ]:
##选择原始 counts 矩阵，并检查它像不像真正的原始计数
if counts_layer in adata.layers:
    adata.X=adata.layers[counts_layer].copy()
    print("use layer:",counts_layer)
else:
    print("layer not found,use adata.X:",counts_layer)
adata.layers["counts"]=adata.X.copy()
x=adata.layers["counts"]
v=x.data if sp.issparse(x) else np.ravel(x)
v=v[:min(len(v),1000000)]
print("counts min",np.min(v),"max",np.max(v),"integer-like",np.allclose(v,np.round(v)))

use layer: rounded_corrected_counts
counts min 1 max 5134 integer-like True


In [ ]:
adata_com_scpoli

AnnData object with n_obs × n_vars = 1018774 × 10
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'barcode_for_match', 'match_id'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    obsp: 'connectivities', 'distances'

In [ ]:
adata

AnnData object with n_obs × n_vars = 1018774 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id'
    var: 'original_gene_names', 'ensembl_id'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'counts'

In [ ]:
#####添加需要的obs
run_tag="precorrect"
for c in ["leiden","cell_type_pred","cell_type_uncert","query","cell_type_pred_ref","cell_type_level1_human","atlas_key"]:
    if c in adata_com_scpoli.obs:
        adata.obs[c]=adata_com_scpoli.obs[c].values
        print("+obs",c)
X=adata_com_scpoli.X.toarray() if sp.issparse(adata_com_scpoli.X) else np.asarray(adata_com_scpoli.X)
adata.obsm["X_scPoli"]=X
if "X_umap" in adata_com_scpoli.obsm:
    adata.obsm["X_umap"]=np.asarray(adata_com_scpoli.obsm["X_umap"])
print("X_scPoli",adata.obsm["X_scPoli"].shape)
print("X_umap",adata.obsm["X_umap"].shape if "X_umap" in adata.obsm else None)
adata.write(os.path.join(outdir,f"allhuman_raw_counts_scpoli_{run_tag}.h5ad"))

+obs leiden
+obs cell_type_pred
+obs cell_type_uncert
+obs query
+obs cell_type_pred_ref
+obs cell_type_level1_human
+obs atlas_key
X_scPoli (1018774, 10)
X_umap (1018774, 2)


In [ ]:
##合并后先画一张全体细胞的 UMAP，检查 cell_type_pred 是否正常
if "X_umap" in adata.obsm and "cell_type_pred" in adata.obs:
    sc.pl.umap(adata,color="cell_type_pred",legend_loc="on data",frameon=False,size=1,show=False)
    plt.savefig(os.path.join(outdir,"umap_cell_type_pred_full.pdf"),bbox_inches="tight")
    plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [ ]:
adata

AnnData object with n_obs × n_vars = 1018774 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_pred_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'counts'

In [ ]:
adata_query = adata[adata.obs['atlas_key']=='query']

In [ ]:
adata_query

View of AnnData object with n_obs × n_vars = 548208 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_pred_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'counts'

In [ ]:
adata_query.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   183446
B cell                    56046
Macrophage                55696
Natural killer cell       51294
Fibroblast                49419
Smooth muscle cell        42831
Endothelial cell          30615
Dendritic cell            24663
Monocyte                  16448
Neutrophil                14157
Pericyte                   9864
Basophil                   6162
Mast cell                  4427
Erythrocyte/Erythroid      3140
Name: count, dtype: int64

In [ ]:
adata_query.obs['cell_type_pred'].value_counts()

cell_type_pred
T cell                   183446
B cell                    56046
Macrophage                55696
Natural killer cell       51294
Fibroblast                49419
Smooth muscle cell        42831
Endothelial cell          30615
Dendritic cell            24663
Monocyte                  16448
Neutrophil                14157
Pericyte                   9864
Basophil                   6162
Mast cell                  4427
Erythrocyte/Erythroid      3140
Name: count, dtype: int64

### mac/mono/dc/neu

In [2]:
adata = sc.read_h5ad("./output_allhuman/allhuman_raw_counts_scpoli_precorrect.h5ad")
adata

AnnData object with n_obs × n_vars = 1018774 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [3]:
##只保留要校正的细胞类型--这里根据cell_type_pred来选
mac_mono_dc_neu_cells=["Macrophage",'Monocyte','Dendritic cell','Neutrophil']
if "cell_type_pred" not in adata.obs:raise ValueError("cell_type_pred not found")
mac_mono_dc_neu=adata[adata.obs["cell_type_pred"].astype(str).isin(mac_mono_dc_neu_cells),:].copy()
print(mac_mono_dc_neu)
print(mac_mono_dc_neu.obs["cell_type_pred"].value_counts())

AnnData object with n_obs × n_vars = 110964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
cell_type_pred
Macrophage        55696
Dendritic cell    24663
Monocyte          16448
Neutrophil        14157
Name: count, dtype: int64


In [4]:
###对选取的数据归一化
mac_mono_dc_neu.X=mac_mono_dc_neu.layers["counts"].copy()
mac_mono_dc_neu.layers["counts"]=mac_mono_dc_neu.X.copy()
sc.pp.normalize_total(mac_mono_dc_neu,target_sum=1e4)
sc.pp.log1p(mac_mono_dc_neu)
mac_mono_dc_neu.layers["lognorm"]=mac_mono_dc_neu.X.copy()
x=mac_mono_dc_neu.layers["lognorm"]
v=x.data if sp.issparse(x) else np.ravel(x)
v=v[:min(len(v),1000000)]
print("lognorm min",np.min(v),"max",np.max(v),"integer-like should be False",np.allclose(v,np.round(v)))

normalizing counts per cell
    finished (0:00:21)
lognorm min 0.061395705 max 7.7495246 integer-like should be False False


In [5]:
mac_mono_dc_neu

AnnData object with n_obs × n_vars = 110964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'log1p'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'lognorm'

In [6]:
mac_mono_dc_neu.obs['cell_type_pred'].value_counts()

cell_type_pred
Macrophage        55696
Dendritic cell    24663
Monocyte          16448
Neutrophil        14157
Name: count, dtype: int64

### 正式校正：不使用 `cell_type_ori` 金标准

下面开始的步骤只依赖：
- `cell_type_pred`：scPoli 预测标签
- `cell_type_uncert`：scPoli uncertainty，如果存在
- `X_scPoli`：latent embedding
- marker gene expression：用表达 marker 做 sanity check

不会再用 `cell_type_ori` / `cell_type_level1` 作为金标准来过滤。


In [3]:
label_col = "cell_type_pred"
uncert_col = "cell_type_uncert"
cluster_key = "leiden_scpoli_res3"

In [ ]:
# 工作对象：前面已经按 cell_type_pred 选出 Macrophage / Monocyte / Dendritic cell / Neutrophil
work = mac_mono_dc_neu.copy()
if label_col not in work.obs:
    raise ValueError(f"{label_col} not found")
if "X_scPoli" not in work.obsm:
    raise ValueError("X_scPoli not found")

print("work object:", work)
print(work.obs[label_col].value_counts())


work object: AnnData object with n_obs × n_vars = 110964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'log1p'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'lognorm'
cell_type_pred
Macrophage        55696
Dendritic cell    24663
Monocyte          16448
Neutrophil        14157
Name: count, dtype: int64


In [10]:
# # 基于 scPoli latent 重新建图、聚类、UMAP
# # 注意：这里不使用 cell_type_ori，只用 X_scPoli 空间结构
# sc.pp.neighbors(work, use_rep="X_scPoli", n_neighbors=150, metric="euclidean")
# sc.tl.leiden(work, resolution=3, key_added=cluster_key, random_state=1234)
# sc.tl.umap(work, min_dist=1.1, random_state=1234)

for c in [cluster_key, label_col]:
    sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=2, show=False)
    plt.savefig(os.path.join(outdir, f"umap_mac_mono_dc_neu_{c}.pdf"), bbox_inches="tight")
    plt.close()

if "dataset" in work.obs:
    sc.pl.umap(work, color="dataset", frameon=False, size=2, show=False)
    plt.savefig(os.path.join(outdir, "umap_mac_mono_dc_neu_dataset.pdf"), bbox_inches="tight")
    plt.close()

print(work.obs[cluster_key].value_counts().sort_index())


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

leiden_scpoli_res3
0     3998
1     3846
2     3701
3     3562
4     3513
5     3407
6     3171
7     3052
8     3019
9     3016
10    2998
11    2990
12    2966
13    2942
14    2921
15    2821
16    2811
17    2746
18    2602
19    2573
20    2516
21    2456
22    2451
23    2435
24    2412
25    2333
26    2255
27    2254
28    2253
29    2212
30    2142
31    1988
32    1906
33    1867
34    1699
35    1695
36    1619
37    1492
38    1490
39    1487
40    1379
41    1193
42    1142
43    1030
44     968
45     588
46     506
47     416
48     125
Name: count, dtype: int64


In [11]:
# 保存包含 scPoli latent、neighbors、Leiden cluster、UMAP 的 AnnData
save_path = os.path.join(outdir, "mac_mono_dc_neu_scPoli_recluster_umap.h5ad")
work.write_h5ad(save_path)

print(f"Saved to: {save_path}")

Saved to: ./output_allhuman/mac_mono_dc_neu_scPoli_recluster_umap.h5ad


In [4]:
work = sc.read_h5ad("./output_allhuman/mac_mono_dc_neu_scPoli_recluster_umap.h5ad")

In [5]:
# local purity：看每个细胞在 X_scPoli 空间的邻居是否和自己预测标签一致
# 没有金标准时，低 local purity 的细胞更可能是边界/混杂/不稳定预测
emb = work.obsm["X_scPoli"]
labels = work.obs[label_col].astype(str).values

K = min(30, work.n_obs - 1)
nn = NearestNeighbors(n_neighbors=K + 1, metric="euclidean").fit(emb)
idx = nn.kneighbors(emb, return_distance=False)[:, 1:]

purity = np.array([
    np.mean(labels[idx[i]] == labels[i])
    for i in range(work.n_obs)
])

work.obs["local_purity"] = purity
print(pd.Series(purity).describe())

# 每类的 local purity 分布，方便你调阈值
purity_summary = (
    work.obs.assign(local_purity=work.obs["local_purity"].astype(float))
    .groupby(label_col, observed=False)["local_purity"]
    .describe()
)
print(purity_summary)
purity_summary.to_csv(os.path.join(outdir, "mac_mono_dc_neu_local_purity_by_pred.csv"))


count    110964.000000
mean          0.915323
std           0.180959
min           0.000000
25%           0.933333
50%           1.000000
75%           1.000000
max           1.000000
dtype: float64
                  count      mean       std  min       25%       50%  75%  max
cell_type_pred                                                                
Dendritic cell  24663.0  0.842773  0.232780  0.0  0.766667  0.966667  1.0  1.0
Macrophage      55696.0  0.944809  0.140926  0.0  0.966667  1.000000  1.0  1.0
Monocyte        16448.0  0.893489  0.201876  0.0  0.900000  1.000000  1.0  1.0
Neutrophil      14157.0  0.951075  0.149576  0.0  1.000000  1.000000  1.0  1.0


In [6]:
# uncertainty：如果存在 cell_type_uncert，用每个预测类型自己的 95% 分位数，来标记高不确定性的细胞
# 这样不需要金标准，也避免一个固定阈值对所有类型过严/过松
if uncert_col in work.obs:
    u = pd.to_numeric(work.obs[uncert_col], errors="coerce")
    work.obs[uncert_col] = u

    uncert_thresh_by_type = (
        work.obs.groupby(label_col, observed=False)[uncert_col]
        .quantile(0.95)
        .to_dict()
    )

    work.obs["uncert_thresh"] = work.obs[label_col].astype(str).map(uncert_thresh_by_type).astype(float)
    work.obs["high_uncert"] = work.obs[uncert_col] > work.obs["uncert_thresh"]

    print("uncertainty threshold by predicted cell type:")
    print(pd.Series(uncert_thresh_by_type).sort_index())
    pd.Series(uncert_thresh_by_type).to_csv(os.path.join(outdir, "mac_mono_dc_neu_uncert_thresh_by_type.csv"))

    sc.pl.umap(work, color=uncert_col, frameon=False, size=2, show=False)
    plt.savefig(os.path.join(outdir, "umap_mac_mono_dc_neu_cell_type_uncert.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["high_uncert"] = False
    print("cell_type_uncert not found; skip uncertainty filtering")


uncertainty threshold by predicted cell type:
Dendritic cell    0.132433
Macrophage        0.164693
Monocyte          0.119345
Neutrophil        0.634439
dtype: float64


In [7]:
work.var

,original_gene_names,ensembl_id
ENSG00000121410,A1BG,ENSG00000121410
ENSG00000268895,A1BG-AS1,ENSG00000268895
ENSG00000148584,A1CF,ENSG00000148584
ENSG00000175899,A2M,ENSG00000175899
ENSG00000245105,A2M-AS1,ENSG00000245105
...,...,...
ENSG00000070476,ZXDC,ENSG00000070476
ENSG00000203995,ZYG11A,ENSG00000203995
ENSG00000162378,ZYG11B,ENSG00000162378
ENSG00000159840,ZYX,ENSG00000159840


In [8]:
work.var_names = work.var['original_gene_names'].values
work.var_names

Index(['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A3GALT2',
       'A4GALT', 'A4GNT', 'AAAS',
       ...
       'ZW10', 'ZWILCH', 'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11B',
       'ZYX', 'ZZEF1'],
      dtype='object', length=28868)

In [9]:
# marker-based sanity check----用marker来给每个细胞打上标签
marker_dict = {
    ##gpt
    # "Macrophage": ['C1QA', 'C1QB', 'C1QC', 'APOE', 'APOC1', 'CD68', 'CD163', 'MRC1', 'MARCO', 'FOLR2', 'TREM2'],
    # "Monocyte": ['CD14', 'FCN1', 'VCAN', 'S100A8', 'S100A9', 'S100A12', 'LYZ', 'CCR2', 'CTSS', 'CSF1R'],
    # "Dendritic cell": ['CD1C', 'CLEC10A', 'FCER1A', 'ITGAX', 'HLA-DRA', 'HLA-DPA1', 'CD74'],
    # "Neutrophil": ['MPO', 'ELANE', 'LCN2', 'CEACAM8', 'CSF3R', 'FCGR3B', 'CXCR2', 'S100A8', 'S100A9'],
    ##manual
    'Macrophage':['C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'],##CD163 from cc,del CD14
    'Monocyte': ['FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'],## CD14 from CC
    'Dendritic cell': ['CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'],
    'Neutrophil': ['NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'],

}

# 只保留对象中存在的 marker
marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    if layer in adata.layers:
        X = adata[:, genes].layers[layer]
    else:
        X = adata[:, genes].X
    m = X.mean(axis=1)
    return np.asarray(m).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]

    # marker_top 和 cell_type_pred 不一致，并且 margin 比较明显，标记为需要 review
    # 这个阈值是经验值，建议结合 dotplot/umap 再调
    work.obs["marker_disagree"] = (
        (work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) &
        (work.obs["marker_margin"].astype(float) > 0.1)
    )

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_mono_dc_neu_by_pred.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_mono_dc_neu_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")


marker genes used:
Macrophage ['C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3', 'CD163']
Monocyte ['FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS', 'CD14']
Dendritic cell ['CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1']
Neutrophil ['NAMPT', 'IFITM2', 'G0S2', 'CXCL8', 'NEAT1', 'SRGN', 'AQP9', 'SOD2', 'FCGR3B', 'IVNS1ABP']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [10]:
# 主要校正依据：cluster-level 校正：不用金标准，只看每个 scPoli cluster 内的预测标签多数派 + marker 多数派
pred_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs[label_col].astype(str))
pred_frac = pred_tab.div(pred_tab.sum(axis=1), axis=0)

cluster_major_pred = pred_tab.idxmax(axis=1)
cluster_major_pred_frac = pred_frac.max(axis=1)

if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
else:
    cluster_major_marker = pd.Series("Unknown", index=pred_tab.index)
    cluster_major_marker_frac = pd.Series(np.nan, index=pred_tab.index)

cluster_summary = pd.DataFrame({
    "cluster": pred_tab.index.astype(str),
    "major_pred": cluster_major_pred.values,
    "major_pred_frac": cluster_major_pred_frac.values,
    "major_marker": cluster_major_marker.reindex(pred_tab.index).values,
    "major_marker_frac": cluster_major_marker_frac.reindex(pred_tab.index).values,
    "n_cells": pred_tab.sum(axis=1).values,
}).set_index("cluster")

# cluster 判定规则：
# 1. 如果预测多数派 >= 0.60 且 marker 多数派一致，则认为该 cluster 可信；
# 2. 如果没有 marker 可用，但预测多数派 >= 0.80，也暂时认为可信；
# 3. 其他 cluster 标为 Uncertain，后续人工看 dotplot/UMAP 再决定。
cluster_summary["cluster_label_clean"] = "Uncertain"

has_marker = cluster_summary["major_marker"].astype(str) != "Unknown"

m1 = (
    (cluster_summary["major_pred_frac"] >= 0.60) &
    has_marker &
    (cluster_summary["major_pred"].astype(str) == cluster_summary["major_marker"].astype(str))
)
cluster_summary.loc[m1, "cluster_label_clean"] = cluster_summary.loc[m1, "major_pred"]

m2 = (
    (cluster_summary["major_pred_frac"] >= 0.80) &
    (~has_marker)
)
cluster_summary.loc[m2, "cluster_label_clean"] = cluster_summary.loc[m2, "major_pred"]

print(cluster_summary.sort_values(["cluster_label_clean", "major_pred_frac"], ascending=[True, False]).to_string())
cluster_summary.to_csv(os.path.join(outdir, "mac_mono_dc_neu_cluster_correction_summary.csv"))


             major_pred  major_pred_frac    major_marker  major_marker_frac  n_cells cluster_label_clean
cluster                                                                                                 
3        Dendritic cell         0.999719  Dendritic cell           0.979787     3562      Dendritic cell
37       Dendritic cell         0.999330  Dendritic cell           0.988606     1492      Dendritic cell
30       Dendritic cell         0.997666  Dendritic cell           0.920635     2142      Dendritic cell
42       Dendritic cell         0.989492  Dendritic cell           0.920315     1142      Dendritic cell
41       Dendritic cell         0.947192  Dendritic cell           0.367980     1193      Dendritic cell
40       Dendritic cell         0.746193  Dendritic cell           0.849166     1379      Dendritic cell
16           Macrophage         1.000000      Macrophage           0.959445     2811          Macrophage
17           Macrophage         1.000000      Macrophag

In [ ]:
# cell-level clean label--
# 先用 scPoli 原始预测；
# 逻辑：
# 先用 scPoli 原始预测；
# 如果 cluster-level 判断很可信，就用 cluster 的 clean label 校正；
# 如果 local_purity 低、uncertainty 高、marker 明显不一致，则标为 Uncertain / needs_review；
# 不覆盖原始 cell_type_pred，保留 cell_type_pred_clean 作为新结果。
work.obs["cell_type_pred_clean"] = work.obs[label_col].astype(str)

cluster_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs["cluster_label_clean"] = work.obs[cluster_key].astype(str).map(cluster_map).fillna("Uncertain")

conf_cluster = work.obs["cluster_label_clean"].astype(str) != "Uncertain"
work.obs.loc[conf_cluster, "cell_type_pred_clean"] = work.obs.loc[conf_cluster, "cluster_label_clean"].astype(str)

# local purity 阈值：四个大类先统一用 0.30；可以根据上面的 purity_summary 再调
purity_thresh = {
    "Macrophage": 0.30,
    "Monocyte": 0.30,
    "Dendritic cell": 0.30,
    "Neutrophil": 0.30,
}
work.obs["purity_thresh"] = work.obs[label_col].astype(str).map(purity_thresh).fillna(0.30).astype(float)
work.obs["low_purity"] = work.obs["local_purity"].astype(float) < work.obs["purity_thresh"]

work.obs["needs_review"] = (
    work.obs["low_purity"].astype(bool) |
    work.obs["high_uncert"].astype(bool) |
    work.obs["marker_disagree"].astype(bool) |
    (work.obs["cluster_label_clean"].astype(str) == "Uncertain")
)

# 对明显不稳定的细胞先标 Uncertain，不硬分
work.obs.loc[work.obs["needs_review"], "cell_type_pred_clean"] = "Uncertain"

print("Original prediction:")
print(work.obs[label_col].value_counts())

print("\nCleaned prediction:")
print(work.obs["cell_type_pred_clean"].value_counts())

clean_summary = pd.crosstab(work.obs[label_col], work.obs["cell_type_pred_clean"])
print(clean_summary)
clean_summary.to_csv(os.path.join(outdir, "mac_mono_dc_neu_pred_vs_clean.csv"))

review_summary = (
    work.obs.groupby(label_col, observed=False)[["low_purity", "high_uncert", "marker_disagree", "needs_review"]]
    .sum()
)
review_summary["before"] = work.obs[label_col].value_counts().reindex(review_summary.index)
review_summary["pct_needs_review"] = review_summary["needs_review"] / review_summary["before"] * 100
print(review_summary)
review_summary.to_csv(os.path.join(outdir, "mac_mono_dc_neu_review_summary.csv"))


In [ ]:
# 可视化 clean label 和 review flags
for c in ["cell_type_pred_clean", "local_purity", "low_purity", "high_uncert", "marker_disagree", "needs_review", cluster_key]:
    if c in work.obs:
        sc.pl.umap(work, color=c, legend_loc="on data" if c in ["cell_type_pred_clean", cluster_key] else "right margin",
                   frameon=False, size=2, show=False)
        plt.savefig(os.path.join(outdir, f"umap_mac_mono_dc_neu_{c}.pdf"), bbox_inches="tight")
        plt.close()

# 只保留非 Uncertain 的 clean 对象；如果你想保留全部细胞，直接使用 work
mac_mono_dc_neu_clean = work[work.obs["cell_type_pred_clean"].astype(str) != "Uncertain", :].copy()

print("before clean:", work.n_obs)
print("after removing Uncertain:", mac_mono_dc_neu_clean.n_obs)
print("removed / uncertain:", work.n_obs - mac_mono_dc_neu_clean.n_obs)
print(mac_mono_dc_neu_clean.obs["cell_type_pred_clean"].value_counts())


In [ ]:
# 把校正结果写回总对象 adata
# 注意：这里写回的是新列，不覆盖原始 cell_type_pred / cell_type_level1
for col in [
    "cell_type_pred_clean",
    "local_purity",
    "high_uncert",
    "low_purity",
    "marker_top",
    "marker_margin",
    "marker_disagree",
    "needs_review",
    "cluster_label_clean",
    cluster_key,
]:
    if col in work.obs:
        adata.obs[col] = np.nan
        adata.obs.loc[work.obs_names, col] = work.obs[col].values
        print("+ write back:", col)

# 如果后面你只想用可信细胞，可以按 needs_review 或 cell_type_pred_clean 过滤
adata.write(os.path.join(outdir, f"allhuman_raw_counts_scpoli_{run_tag}_mac_mono_dc_neu_corrected_no_ori.h5ad"))
work.write(os.path.join(outdir, f"mac_mono_dc_neu_before_filter_{run_tag}_no_ori.h5ad"))
mac_mono_dc_neu_clean.write(os.path.join(outdir, f"mac_mono_dc_neu_clean_{run_tag}_no_ori.h5ad"))

print("done")
print("outdir:", outdir)
